In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# Load IPC dataset
df = pd.read_excel("ipcphase.xlsx")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [10]:
# Create a working copy
ipc = df.copy()

print("Working copy created successfully.")

Working copy created successfully.


In [11]:
# Standardize the Column Names

ipc.columns = (
    ipc.columns
       .str.strip()
       .str.lower()
       .str.replace(" ", "_")
)

ipc.columns.tolist()

['row',
 'source_organization',
 'source_document',
 'country',
 'country_code',
 'geographic_group',
 'fewsnet_region',
 'geographic_unit_full_name',
 'geographic_unit_name',
 'unit_type',
 'fnid',
 'classification_scale',
 'scenario_name',
 'preference_rating',
 'is_allowing_for_assistance',
 'projection_start',
 'projection_end',
 'status',
 'value',
 'pct_phase3',
 'pct_phase4',
 'pct_phase5',
 'description',
 'id',
 'datacollectionperiod',
 'datacollection',
 'scenario',
 'geographic_unit',
 'datasourceorganization',
 'datasourcedocument',
 'dataseries',
 'dataseries_name',
 'specialization_type',
 'dataseries_specialization_type',
 'data_usage_policy',
 'created',
 'modified',
 'status_changed',
 'collection_status',
 'collection_status_changed',
 'collection_schedule',
 'reporting_date']

In [12]:
#Remove columns with 100% missing Values
cols_to_drop = [
    'pct_phase3',
    'pct_phase4',
    'pct_phase5'
]

ipc.drop(columns=cols_to_drop, inplace=True)

print(ipc.shape)

(27694, 39)


In [13]:
# Remove Administrative Metadata
metadata_cols = [

    'row',
    'id',

    'created',
    'modified',
    'status_changed',

    'collection_status',
    'collection_status_changed',
    'collection_schedule',

    'datasourceorganization',
    'datasourcedocument',

    'data_usage_policy',

    'specialization_type',
    'dataseries_specialization_type'
]

ipc.drop(columns=metadata_cols, inplace=True)

print(ipc.shape)

(27694, 26)


In [14]:
# Convert Date Variables
date_cols = [

    'projection_start',
    'projection_end',
    'reporting_date'

]

for col in date_cols:
    ipc[col] = pd.to_datetime(ipc[col])

ipc[date_cols].dtypes

projection_start    datetime64[us]
projection_end      datetime64[us]
reporting_date      datetime64[us]
dtype: object

In [15]:
#check for duplicate records
duplicates = ipc.duplicated().sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [16]:
# Check for missing values
missing = (
    ipc.isnull()
       .sum()
       .sort_values(ascending=False)
)

missing[missing > 0]

Series([], dtype: int64)

In [17]:
#Verify IPC Phases
ipc[['value', 'description']] \
    .drop_duplicates() \
    .sort_values('value')

,value,description
0,1.0,Minimal
152,2.0,Stressed
172,3.0,Crisis
323,4.0,Emergency


In [18]:
# Check scenario
ipc['scenario_name'].value_counts()

scenario_name
Current Situation    27694
Name: count, dtype: int64

In [19]:
#simply dataset as all of the obsevartions belong to the same dataset
ipc.drop(columns=['scenario_name'], inplace=True)

In [20]:
#Check country
ipc['country'].value_counts()

country
Kenya    27694
Name: count, dtype: int64

In [21]:
#drop columns that dont add any predictive value sinceevery record is in Kenya
ipc.drop(columns=['country'], inplace=True)
ipc.drop(columns=['country_code'], inplace=True)

In [22]:
#Check constant columns
constant_cols = []

for col in ipc.columns:
    if ipc[col].nunique() == 1:
        constant_cols.append(col)

constant_cols

['source_organization',
 'source_document',
 'geographic_group',
 'fewsnet_region',
 'preference_rating',
 'status',
 'scenario']

In [23]:
#If a column has only one unique value across all records, it doesn't help distinguish observations and can usually be removed.
ipc.drop(columns=constant_cols, inplace=True)

print(ipc.shape)

(27694, 16)


In [24]:
#Sort chronologically
ipc.sort_values(
    by=['geographic_unit_name', 'reporting_date'],
    inplace=True
)

ipc.reset_index(drop=True, inplace=True)

In [26]:
print("FINAL DATASET SUMMARY")


print(f"Rows: {ipc.shape[0]:,}")
print(f"Columns: {ipc.shape[1]}")

print("\nMissing Values")
print(ipc.isnull().sum().sum())

print("\nDuplicates")
print(ipc.duplicated().sum())

FINAL DATASET SUMMARY
Rows: 27,694
Columns: 16

Missing Values
0

Duplicates
0


In [31]:
ipc.drop(
    columns=[
        'datacollectionperiod',
        'datacollection',
        'geographic_unit',
        'dataseries'
    ],
    inplace=True
)

print(ipc.shape)

(27694, 12)


In [32]:
# Save the Cleaned Dataset
ipc.to_csv("ipc_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


Why i kept certain Variables:


| Variable                     | Reason                                                          |
| ---------------------------- | --------------------------------------------------------------- |
| `value`                      | Target variable (IPC phase)                                     |
| `description`                | Human-readable phase labels for interpretation                  |
| `geographic_unit_name`       | Geographic identifier for grouping and panel analysis           |
| `unit_type`                  | Distinguishes administrative and livelihood-zone units          |
| `projection_start`           | Time feature                                                    |
| `projection_end`             | Time feature                                                    |
| `reporting_date`             | Primary time index                                              |
| `classification_scale`       | IPC version; may be included as a feature or for stratification |
| `is_allowing_for_assistance` | Potential explanatory variable                                  |
| `preference_rating`          | Data quality/priority indicator                                 |
| `fnid`                       | Unique geographic identifier if needed for grouping             |


In [33]:
ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 27694 entries, 0 to 27693
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   geographic_unit_full_name   27694 non-null  str           
 1   geographic_unit_name        27694 non-null  str           
 2   unit_type                   27694 non-null  str           
 3   fnid                        27694 non-null  str           
 4   classification_scale        27694 non-null  str           
 5   is_allowing_for_assistance  27694 non-null  bool          
 6   projection_start            27694 non-null  datetime64[us]
 7   projection_end              27694 non-null  datetime64[us]
 8   value                       27694 non-null  float64       
 9   description                 27694 non-null  str           
 10  dataseries_name             27694 non-null  str           
 11  reporting_date              27694 non-null  datetime64[us]
dtypes

In [29]:
print(ipc.columns.tolist())

['geographic_unit_full_name', 'geographic_unit_name', 'unit_type', 'fnid', 'classification_scale', 'is_allowing_for_assistance', 'projection_start', 'projection_end', 'value', 'description', 'datacollectionperiod', 'datacollection', 'geographic_unit', 'dataseries', 'dataseries_name', 'reporting_date']


In [30]:
for col in [
    'datacollectionperiod',
    'datacollection',
    'geographic_unit',
    'dataseries'
]:
    print(f"\n{col}")
    print(ipc[col].nunique())
    print(ipc[col].head())


datacollectionperiod
49
0    160280
1    160283
2    160286
3    160289
4    160292
Name: datacollectionperiod, dtype: int64

datacollection
49
0    168956
1    168957
2    168958
3    168959
4    168960
Name: datacollection, dtype: int64

geographic_unit
1218
0    22989
1    22989
2    22989
3    22989
4    22989
Name: geographic_unit, dtype: int64

dataseries
2897
0    6515211
1    6515211
2    6515211
3    6515211
4    6515211
Name: dataseries, dtype: int64
